# 02 · Clean the data and first emotion pass

**Phase 3 deliverables:** a clean dataset with its final count per brand, a 200-post emotion sample, a 20-post hand check, and your final 5–6 core emotion groups.

Run the cells in order. The model cell downloads about 500 MB the first time.

*Independent student project, not affiliated with WHOOP.*

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import clean_data, emotions

pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 200)

## 1 · Clean

Removes duplicates, deleted posts, bot replies, anything under 15 words, and non-English text. The funnel shows how many rows survive each step. It's a good table for the method section of your write-up.

In [ ]:
raw = pd.read_csv(clean_data.RAW)
clean, funnel = clean_data.clean(raw)
clean.to_csv(clean_data.CLEAN, index=False)
funnel.to_csv(clean_data.FUNNEL)
funnel

### Final clean counts

**The "All brands" total is the number for your resume** (posts and comments analyzed). Use the `random` column for overall brand comparisons in Phase 4.

In [ ]:
final = clean_data.final_counts(clean)
final

## 2 · Load the emotion model

In [ ]:
clf = emotions.load_model()

## 3 · Label a 200-post sample

50 random-sample rows per brand, fixed seed so you get the same sample every time.

In [ ]:
pool = clean[clean["sample"] == "random"]
n = min(50, pool["brand"].value_counts().min())
sample = pool.groupby("brand").sample(n=n, random_state=42).reset_index(drop=True)

scores = emotions.score_texts(clf, sample["text_model"])
labeled = pd.concat([sample.reset_index(drop=True), emotions.add_core_emotions(scores)], axis=1)
labeled.to_csv(ROOT / "data" / "clean" / "emotion_sample_200.csv", index=False)
print(f"Labeled {len(labeled)} rows")

### How often does each of the 28 labels show up?

Use this to decide your core groups. Watch for:
- **neutral**: Reddit is full of factual questions, so a high neutral share is normal.
- **Big labels outside your groups**, for example *confusion*, *gratitude*, *approval* or *admiration*. If one is large and relevant, consider adding it as a group or merging it into one.

In [ ]:
dist = emotions.label_distribution(scores)
dist.style.format({"share_above_threshold": "{:.0%}", "share_as_top_label": "{:.0%}"})

### Core emotion mix in the sample, by brand

In [ ]:
flags = [c for c in labeled.columns if c.startswith("expresses_")]
mix = labeled.groupby("brand")[flags].mean().rename(columns=lambda c: c.replace("expresses_", ""))
mix["none (neutral/other)"] = labeled.groupby("brand")["primary_emotion"].apply(lambda s: (s == "none").mean())
mix.style.format("{:.0%}")

## 4 · Hand-check 20 posts

Five per brand. Read each one and fill in the last two columns in the CSV (`your_emotion`, `model_ok` as y/n), then come back and run the next cell. Aim for at least 15 of 20 that make sense. If many are wrong in the same way, the threshold or the groups need adjusting.

The file has post text, so it lives in `data/clean/` and never goes to GitHub.

In [ ]:
check = (labeled.groupby("brand").sample(n=5, random_state=7)
         [["brand", "type", "text_model", "top_labels", "primary_emotion"]]
         .assign(your_emotion="", model_ok=""))
check_path = ROOT / "data" / "clean" / "handcheck_20.csv"
check.to_csv(check_path, index=False)
print(f"Saved {check_path.relative_to(ROOT)}. Open it in Excel or Numbers and fill in the last two columns.")
check[["brand", "text_model", "top_labels", "primary_emotion"]]

In [ ]:
# Run after filling in the CSV
done = pd.read_csv(check_path)
ok = done["model_ok"].astype(str).str.strip().str.lower().str.startswith("y")
print(f"Model labels made sense for {ok.sum()} of {len(done)} posts ({ok.mean():.0%})")

## 5 · Phase 3 done when

- [ ] Final clean count per brand recorded (the table in section 1)
- [ ] 20 hand-checked posts, at least 15 make sense
- [ ] Core emotion groups decided. If you change them, edit `CORE_GROUPS` in `src/emotions.py` and re-run sections 3–4.
